In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

np.random.seed(42)

# Load Heart Disease Dataset
print("Loading Heart Disease Dataset...")
try:
    data = fetch_openml(name="heart-statlog", version=1, as_frame=True)
    df = data.frame.copy()
    X_df = df.drop(columns=[data.target_names[0]], errors="ignore")
    y_series = data.target
    if y_series is None:
        y_series = df.iloc[:, -1]
    X_df = X_df.select_dtypes(include=[np.number])
    y_raw = pd.Series(y_series).astype(str)
    y = y_raw.apply(
        lambda v: 0 if v.strip().lower() in ["0", "absence", "absent", "negative", "no", "1"] else 1
    ).values
    if len(np.unique(y)) != 2:
        raise ValueError("Could not identify binary target.")
    X = X_df.values.astype(float)
    feature_names = list(X_df.columns)
except Exception as e:
    print("OpenML loading failed:", e)
    print("Trying the UCI Heart Disease dataset...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    columns = [
        "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
    ]
    df = pd.read_csv(url, names=columns, na_values="?")
    df = df.dropna()
    X = df.drop(columns="target").values.astype(float)
    y = (df["target"].values > 0).astype(int)
    feature_names = columns[:-1]

# Preprocess and normalize
imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# MADALINE Network
class MADALINE:
    def __init__(self, n_units=5, learning_rate=0.01, epochs=300):
        self.n_units = n_units
        self.lr = learning_rate
        self.max_epochs = epochs
        self.weights = None
        self.biases = None
        self.epochs = 0

    def unit_outputs(self, X):
        return X @ self.weights.T + self.biases

    def unit_decisions(self, X):
        return (self.unit_outputs(X) >= 0).astype(int)

    def predict(self, X):
        decisions = self.unit_decisions(X)
        return (decisions.sum(axis=1) > self.n_units / 2).astype(int)

    def fit(self, X, y):
        n_features = X.shape[1]
        self.weights = np.random.normal(0, 0.1, (self.n_units, n_features))
        self.biases = np.zeros(self.n_units)

        targets = np.where(y == 1, 1.0, -1.0)

        for epoch in range(self.max_epochs):
            for xi, target in zip(X, targets):
                outputs = self.weights @ xi + self.biases
                network_output = 1 if np.sum(outputs >= 0) > self.n_units / 2 else -1

                if network_output != target:
                    closest_unit = np.argmin(np.abs(outputs))
                    error = target - outputs[closest_unit]
                    self.weights[closest_unit] += self.lr * error * xi
                    self.biases[closest_unit] += self.lr * error

            self.epochs = epoch + 1

        return self

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y) * 100

# Train default network
model = MADALINE(n_units=5)
model.fit(X_train, y_train)

def class_name(value):
    return "Heart Disease Present" if int(value) == 1 else "No Heart Disease"

def get_vote_info(net, sample):
    outputs = net.unit_outputs(sample.reshape(1, -1))[0]
    decisions = (outputs >= 0).astype(int)
    positive = int(decisions.sum())
    negative = len(decisions) - positive
    final = int(positive > len(decisions) / 2)
    return outputs, decisions, positive, negative, final

def feature_influence(net, sample):
    base = net.predict(sample.reshape(1, -1))[0]
    base_outputs = net.unit_outputs(sample.reshape(1, -1))[0]
    effects = []

    for i, name in enumerate(feature_names):
        changed = sample.copy()
        changed[i] = 0
        new_outputs = net.unit_outputs(changed.reshape(1, -1))[0]
        change = np.mean(np.abs(new_outputs - base_outputs))
        effects.append((name, change))

    return sorted(effects, key=lambda z: z[1], reverse=True)

def build_network(n_units):
    net = MADALINE(n_units=n_units)
    net.fit(X_train, y_train)
    return net

# 1. Heart Disease Classification
def classification():
    preds = model.predict(X_test)
    unit_outputs = model.unit_outputs(X_test)
    unit_decisions = (unit_outputs >= 0).astype(int)

    print("\nHEART DISEASE CLASSIFICATION")
    for i in range(len(y_test)):
        print(
            f"Patient {i+1:3d} | Actual: {class_name(y_test[i]):22s} | "
            f"Predicted: {class_name(preds[i]):22s} | "
            f"Unit Outputs: {np.round(unit_outputs[i], 3)} | "
            f"Majority: {class_name(preds[i])}"
        )

    print(f"\nAccuracy: {model.accuracy(X_test, y_test):.2f}%")

# 2. ADALINE Unit Voting Analyzer
def voting_analyzer():
    unanimous = 0
    majority = 0
    tie_or_near = 0

    print("\nADALINE UNIT VOTING ANALYZER")
    for i, sample in enumerate(X_test):
        outputs, decisions, pos, neg, final = get_vote_info(model, sample)
        if pos == 0 or neg == 0:
            status = "Unanimous"
            unanimous += 1
        elif abs(pos - neg) <= 1:
            status = "Tie/Near-Tie"
            tie_or_near += 1
        else:
            status = "Majority"
            majority += 1

        print(
            f"Patient {i+1:3d} | Votes: {decisions} | Positive: {pos} | "
            f"Negative: {neg} | Decision: {class_name(final)} | {status}"
        )

    total = len(X_test)
    print(f"\nUnanimous decisions: {unanimous} ({unanimous/total*100:.2f}%)")
    print(f"Majority decisions: {majority} ({majority/total*100:.2f}%)")
    print(f"Tie/Near-Tie decisions: {tie_or_near} ({tie_or_near/total*100:.2f}%)")

# 3. Minimal Disturbance Weight Update
def minimal_update_challenge():
    index = int(input(f"Enter test patient number (1-{len(X_test)}): ")) - 1
    if index < 0 or index >= len(X_test):
        print("Invalid patient number.")
        return

    sample = X_test[index]
    actual = int(y_test[index])
    outputs, decisions, pos, neg, final = get_vote_info(model, sample)

    print("\nBefore update")
    print("Actual:", class_name(actual))
    print("Prediction:", class_name(final))
    print("Unit outputs:", np.round(outputs, 4))

    if final == actual:
        print("Prediction is already correct; no update required.")
        return

    unit = int(np.argmin(np.abs(outputs)))
    old_output = outputs[unit]
    old_weights = model.weights[unit].copy()

    target = 1 if actual == 1 else -1
    error = target - old_output
    model.weights[unit] += model.lr * error * sample
    model.biases[unit] += model.lr * error

    new_output = model.unit_outputs(sample.reshape(1, -1))[0]
    new_prediction = model.predict(sample.reshape(1, -1))[0]

    print("Selected Unit:", unit + 1)
    print("Old Linear Output:", round(old_output, 4))
    print("Old Weights:", np.round(old_weights, 4))
    print("Updated Weights:", np.round(model.weights[unit], 4))
    print("New Unit Outputs:", np.round(new_output, 4))
    print("New MADALINE Decision:", class_name(new_prediction))

# 4. Hidden ADALINE Unit Disagreement
def disagreement_analyzer():
    records = []

    for i, sample in enumerate(X_test):
        outputs, decisions, pos, neg, final = get_vote_info(model, sample)
        disagreement = min(pos, neg) / len(decisions) * 100
        records.append({
            "Patient": i + 1,
            "Positive Votes": pos,
            "Negative Votes": neg,
            "Disagreement (%)": disagreement,
            "Actual": class_name(y_test[i]),
            "Predicted": class_name(final)
        })

    df = pd.DataFrame(records).sort_values("Disagreement (%)", ascending=False)
    print("\nTOP FIVE DISAGREEMENT RECORDS")
    print(df.head(5).to_string(index=False))

    effects = feature_influence(model, X_test[np.argmax(
        [r["Disagreement (%)"] for r in records]
    )])
    print("\nFeatures associated with unit-output changes:")
    for name, value in effects:
        print(f"{name}: {value:.4f}")

# 5. Patient Decision Flip Analyzer
def decision_flip_analyzer():
    patient = int(input(f"Enter test patient number (1-{len(X_test)}): ")) - 1
    if patient < 0 or patient >= len(X_test):
        print("Invalid patient number.")
        return

    print("\nFeatures:")
    for i, name in enumerate(feature_names):
        print(f"{i+1}. {name}")

    feature = int(input(f"Select feature (1-{len(feature_names)}): ")) - 1
    if feature < 0 or feature >= len(feature_names):
        print("Invalid feature.")
        return

    sample = X_test[patient].copy()
    original = int(model.predict(sample.reshape(1, -1))[0])
    print("Original Decision:", class_name(original))

    values = np.linspace(sample[feature] - 3, sample[feature] + 3, 121)
    previous = original
    flips = []

    print("\nNormalized Feature Value | Prediction | Unit Decisions")
    for value in values:
        modified = sample.copy()
        modified[feature] = value
        outputs, decisions, pos, neg, prediction = get_vote_info(model, modified)
        print(f"{value:9.4f} | {class_name(prediction):22s} | {decisions}")

        if prediction != previous:
            flips.append((value, prediction))
        previous = prediction

    if flips:
        print("\nDetected Decision Changes:")
        for value, prediction in flips:
            print(f"Feature value {value:.4f} -> {class_name(prediction)}")
    else:
        print("No decision flip detected in the tested range.")

# 6. Network Size Challenge
def network_size_challenge():
    results = []

    for n in [3, 5, 7, 9, 11]:
        net = build_network(n)
        preds = net.predict(X_test)
        unit_decisions = net.unit_decisions(X_test)
        unanimous = np.sum(
            (unit_decisions.sum(axis=1) == 0) |
            (unit_decisions.sum(axis=1) == n)
        )
        conflicting = len(X_test) - unanimous
        votes = unit_decisions.sum(axis=1)
        ambiguous = np.sum(np.abs(2 * votes - n) <= 1)

        results.append({
            "Units": n,
            "Epochs": net.epochs,
            "Accuracy (%)": net.accuracy(X_test, y_test),
            "Unanimous": unanimous,
            "Conflicting": conflicting,
            "Ambiguous": ambiguous,
            "Misclassified": int(np.sum(preds != y_test))
        })

    df = pd.DataFrame(results)
    print("\nMADALINE NETWORK SIZE COMPARISON")
    print(df.round(3).to_string(index=False))

    best = df.sort_values(
        ["Accuracy (%)", "Misclassified"], ascending=[False, True]
    ).iloc[0]
    print("\nMost Suitable Network Size:", int(best["Units"]), "units")

# 7. Ambiguous Patient Record Detector
def ambiguous_record_detector():
    records = []

    for i, sample in enumerate(X_test):
        outputs, decisions, pos, neg, final = get_vote_info(model, sample)
        records.append({
            "Patient": i + 1,
            "Positive Votes": pos,
            "Negative Votes": neg,
            "Voting Margin": abs(pos - neg),
            "Actual": class_name(y_test[i]),
            "Predicted": class_name(final),
            "Unit Outputs": np.round(outputs, 3).tolist()
        })

    df = pd.DataFrame(records).sort_values("Voting Margin")
    print("\nFIVE MOST AMBIGUOUS PATIENT RECORDS")
    print(df.head(5).to_string(index=False))

# 8. Performance Report
def performance_report():
    results = []

    for n in [3, 5, 7, 9, 11]:
        net = build_network(n)
        preds = net.predict(X_test)
        votes = net.unit_decisions(X_test)
        vote_count = votes.sum(axis=1)

        unanimous = np.sum((vote_count == 0) | (vote_count == n))
        majority = np.sum(np.abs(2 * vote_count - n) > 1)
        ambiguous = np.sum(np.abs(2 * vote_count - n) <= 1)
        conflicting = len(X_test) - unanimous

        results.append({
            "Units": n,
            "Epochs": net.epochs,
            "Accuracy (%)": net.accuracy(X_test, y_test),
            "Unanimous": int(unanimous),
            "Majority": int(majority),
            "Conflicting": int(conflicting),
            "Ambiguous": int(ambiguous),
            "Misclassified": int(np.sum(preds != y_test))
        })

    print("\nMADALINE CLASSIFICATION PERFORMANCE REPORT")
    print(pd.DataFrame(results).round(3).to_string(index=False))

# Menu
def main_menu():
    while True:
        print("\n" + "=" * 65)
        print(" MADALINE HEART DISEASE CLASSIFICATION SYSTEM")
        print("=" * 65)
        print("1. Heart Disease Classification using MADALINE")
        print("2. ADALINE Unit Voting Analyzer")
        print("3. Minimal Disturbance Weight Update Challenge")
        print("4. Hidden ADALINE Unit Disagreement Analyzer")
        print("5. Patient Decision Flip Analyzer")
        print("6. MADALINE Network Size Challenge")
        print("7. Ambiguous Patient Record Detector")
        print("8. MADALINE Classification Performance Report")
        print("9. Exit")

        choice = input("\nEnter your choice (1-9): ").strip()

        if choice == "1":
            classification()
        elif choice == "2":
            voting_analyzer()
        elif choice == "3":
            minimal_update_challenge()
        elif choice == "4":
            disagreement_analyzer()
        elif choice == "5":
            decision_flip_analyzer()
        elif choice == "6":
            network_size_challenge()
        elif choice == "7":
            ambiguous_record_detector()
        elif choice == "8":
            performance_report()
        elif choice == "9":
            print("\nPROGRAM FINISHED")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 9.")

main_menu()

Loading Heart Disease Dataset...

 MADALINE HEART DISEASE CLASSIFICATION SYSTEM
1. Heart Disease Classification using MADALINE
2. ADALINE Unit Voting Analyzer
3. Minimal Disturbance Weight Update Challenge
4. Hidden ADALINE Unit Disagreement Analyzer
5. Patient Decision Flip Analyzer
6. MADALINE Network Size Challenge
7. Ambiguous Patient Record Detector
8. MADALINE Classification Performance Report
9. Exit

Enter your choice (1-9): 9

PROGRAM FINISHED
